---
downloads:
  - id: lecture-script
    title: Lecture Script (.pdf)
  - file: session2_cluster.ipynb
    title: Jupyter Notebook (.ipynb)
  - file: requirements.txt
    title: Requirements (.txt)
  - file: ../downloads/hello-cluster-job.yml
    title: Hello Cluster Job (.yml)
  - file: ../downloads/cluster/storage-pvc.yml
    title: Storage PVC (.yml)
  - file: ../downloads/cluster/Dockerfile
    title: Dockerfile
  - file: ../downloads/cluster/remote-deployment.yml
    title: Remote Deployment (.yml)
---

# Session 2: BHT Cluster and Experiments Logging

⚠️ <span style="color: orange;">Please note we are currently reworking the lecture and still preparing the course materials. You will find this message wherever the course materials are being updated / not finalized.</span> ⚠️

## Learning goals

- Understand cluster concepts for Data Science with Kubernetes
    - Pods, Deployments, Services, Jobs, ...
- Run interactive and batch workloads on our BHT cluster
- Work resource-aware (CPU, RAM, GPU, storage)
- Log experiments with Weights & Biases (WandB) to track while running on the cluster

## BHT cluster

- Multiple nodes with CPUs, RAM, and GPUs for more computing power than your laptops
- Monitor the resources in [Grafana](https://monitoring.cluster.ris.bht-berlin.de)
- This is what it sounds like: [Youtube Video](https://www.youtube.com/shorts/7BtNBv9JK6c)
- Here you find the documentation: [BHT Cluster Documentation](https://docs.cluster.ris.bht-berlin.de/)
- Dashboard for your own workspace: [Headlamp](https://dashboard.cluster.ris.bht-berlin.de/)

Before we start with the tutorial, lets take a look at the GPUs:

![GPU Monitoring in Grafana](../figures/monitoring_gpus.png)

⚠️ Please be mindful of the resources you use on the cluster. PhDs and students are sharing the same cluster, so we need to be considerate of each other.


### Kubernetes

```{image} ../figures/k8s.png
:alt: Kubernetes
:width: 150px
:align: left
```

- Open-source system for managing containerized apps
- Deploys, keeps your apps alive
- Second-largest open-source project after Linux

### Kubernetes core concepts

- **Pod**: Smallest deployable unit that runs docker containers
- **Job**: One-off task that run to completion and then stop
- **Deployment**: Keeps desired number of pods alive, restarts if they fail
- **PVC**: **P**ersistent **V**olume **C**laim, requests persistent storage
- **Secret**: Stores sensitive info (like SSH keys)
- Others: Namespaces, Cronjobs, Services ...

### Docker concepts
```{image} ../figures/docker.png
:alt: Docker
:width: 150px
:align: left
```
- **Image**: Blueprint for a container, includes code and dependencies
- **Container**: Running instance of an image, isolated environment
- **Dockerfile**: Instructions to build an image
- **Docker Hub**: Public registry for sharing images

Example Dockerfile:

```dockerfile
FROM python:3.10-slim # Use a parent image
WORKDIR /app # Set the working directory in the container
COPY . /app # Copy the current directory contents into the container at /app
RUN pip install --no-cache-dir -r requirements.txt # Install packages
CMD ["python", "main.py"] # Run main.py when the container launches
```

### Prerequisites for working on our cluster

You should have done the following already (as described at the end of Session 1):
- BHT account
- Internet access with a VPN connection to BHT (see: https://doku.bht-berlin.de/zugang/vpn)
- Access rights to the cluster (should be taken care of)
- Install `kubectl`: https://kubernetes.io/docs/tasks/tools/
- Configure `kubectl` for BHT login (see: https://docs.cluster.ris.bht-berlin.de/user/quickstart/#direct-kubernetes-access)
- Install `docker`: https://docs.docker.com/get-docker/

*Recommended*: VSCode (Tutorial will show an example for VSCode)

### First Python code on the cluster
**Example: Printing "Hello from the cluster!" using a Job**<br>
**Step 1:** Download the [`Hello Cluster Job (.yml)`](../downloads/hello-cluster-job.yml) file from the downloads section.<br>
**Step 2:** Store it somewhere on your computer and open it with an editor (e.g. VSCode).<br>
**Step 3:** Send it to the cluster:<br>
```bash
kubectl apply -f hello-cluster-job.yml
```
**Step 4:** Check the status of your job:<br>
```bash
kubectl get jobs
```
**Step 5:** Check the logs of your job:<br>
```bash
kubectl logs job/hello-cluster-job
```
You should see the following output:
*Hello from the cluster!*

**Step 6:** Clean up the job from the cluster:<br>
```bash
kubectl delete job hello-cluster-job
```

### Prototyping Workflow: SSH into your pod with VSCode
- You can use VSCode's Remote Development Extension to SSH into your pod and work on the cluster directly from your editor.
- This allows you to run code, edit files, and monitor resources without leaving VSCode.

#### Prerequisites
- A *password-protected* SSH key pair for authentication
- A Kubernetes secret that injects the **public** SSH key into the pod

**Step 1:** Generate a new SSH key pair
```bash
ssh-keygen
```
- Follow the prompts to create a new key pair (e.g., `id_rsa_cluster` and `id_rsa_cluster.pub`)
- Make sure to **set a password** for the private key
- Save the keys in a secure location on your computer (usually in `~/.ssh/`)

**Step 2:** Create a Kubernetes secret with your **public** SSH key (ends with `.pub`)
```bash
kubectl create secret generic ssh-key-secret --from-file=authorized_keys=~/.ssh/id_rsa_cluster.pub
```
- Replace `~/.ssh/id_rsa_cluster.pub` with the actual path to your public SSH key if it's different
- This command creates a secret on the cluster named `ssh-key-secret` that contains your public SSH key
- This can be mounted into your pod to allow SSH access, check it with:
```bash
kubectl get secrets
```
**Step 3:** Create a local SSH config file (if you don't have one already)<br>
For Mac/Linux:
```bash
touch ~/.ssh/config
```
For Windows (PowerShell):
```powershell
New-Item -Path $env:USERPROFILE\.ssh\config -ItemType File
```
Open the `config` file in an editor and add the following configuration:
```
Host bht-cluster
    HostName localhost # We connect through port forwarding
    Port 2222 # This is the port you will forward to your pod
    User root
    IdentityFile ~/.ssh/id_rsa_cluster # Path to your private SSH key
```

Prerequisites are now set up, next we can setup a pod.

#### Setting your own pod (Docker + Kubernetes)
**Step 1:** Claim a persistent volume with a PVC, for storing code and data:
```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: dsw-pvc
spec:
  accessModes:
    - ReadWriteOnce # Use ReadWriteMany if you have multiple Pods needing to write to the volume
  resources:
    requests:
      storage: 5Gi # Start low, you can always increase but NOT decrease the storage size later
```
- Save this as `storage-pvc.yml` and apply it to the cluster:
```bash
kubectl apply -f storage-pvc.yml
```
- This creates a persistent volume claim named `dsw-pvc` that requests 5GB of storage. You can check the status of your PVC with:
```bash
kubectl get pvc
```


**Step 2:** Create a docker image with an SSH server
- Download the [`Dockerfile`](../downloads/cluster/Dockerfile) from the downloads section.
- Navigate to the directory where you downloaded the Dockerfile and build the image:
```bash
docker build -t ssh-server-image .
```
- Start the container locally to test it:
```bash
docker run -p 2222:22 ssh-server-image
```
- This command runs the container and forwards port 22 inside the container to port 2222 on your local machine, allowing you to SSH into it using the configuration we set up earlier.

**Step 3:** Push the image to a container registry
- You need to push your image to a container registry that our cluster can access.
    1. Example: Docker Hub **(PUBLIC)**
        - Name your image with your Docker Hub username:
        ```bash
        docker tag ssh-server-image <your-dockerhub-username>/ssh-server-image:latest
        ```
        - *Optional:* If not already done via e.g. Docker Desktop, login:
        ```bash
        docker login hub.docker.com
        ```
        - Push to Docker Hub:
        ```bash
        docker push <your-dockerhub-username>/ssh-server-image:latest
        ```
    2. For private registries, have a look at the cluster documentation: https://docs.cluster.ris.bht-berlin.de/user/images/

**Step 4:** Create a Kubernetes deployment that uses your image and mounts the PVC
- Download the [`Remote Deployment (.YML)`](../downloads/cluster/remote-deployment.yml) from the downloads section.
- Open the `remote-deployment.yml` file and modify the following parts:
    - Under `containers.image`, replace `your-dockerhub-username/ssh-server-image:latest` with the name of your image in the container registry.
    - Under `volumes`, make sure the `claimName` matches the name of your PVC (e.g., `storage-pvc`).
    - Under `volumes`, make sure the `secretName` matches the name of your SSH key secret (e.g., `ssh-key-secret`).
- Apply the deployment to the cluster:
```bash
kubectl apply -f remote-deployment.yml
```
- Check the status of your deployment:
```bash
kubectl get deploy
```
- Check the status of your pod:
```bash
kubectl get pods
```
- Check whether your pod is using the volume:
```bash
kubectl describe pod <pod-name>
```

Once your pod is running you can proceed and connect to it via SSH.

**Step 5:** Port forward to your pod to enable SSH access
- Kubernetes pods are not directly accessible from outside the cluster. To SSH into your pod, you need to set up port forwarding from your local machine to the pod.
    1. Start **port-forwarding** in a terminal:
    ```bash
    kubectl port-forward <pod-name> 2222:22
    ```
    - Replace `<pod-name>` with the actual name of your pod (you can get it from `kubectl get pods`)
    - This command forwards port 22 in the pod to port 2222 on your local machine.
    - **Keep this terminal running** as long as you want to have SSH access. It's your live bridge to the pod.
    2. Connect to the pod using SSH:
    ```bash
    ssh bht-cluster
    ```
    - This uses the SSH configuration we set up earlier to connect to the pod through the forwarded port.
    - You should be prompted for the password of your SSH key.
 
Congratulations! You are now connected to your pod on the cluster via SSH!


**Step 6:** Connect your VSCode to the pod
- In VSCode download the "Remote Development" extension pack if you haven't already.
- Click on the "Remote Window Icon" ("><") in the bottom left corner or press 'Ctrl+Shift+P' and select "Remote-SSH: Connect to Host...".
- On the dropdown, select "bht-cluster" (or whatever you named your host in the SSH config) and enter the password for your SSH key when prompted.
- Once connected, you might want to install some VSCode extensions in the remote environment (e.g. Python extension) to make your life easier when working on the cluster.
- Often times you need to reload the window. For this you can press 'Ctrl+Shift+P' and select "Reload Window".

You can now open terminals, edit files, and run code on the cluster directly from VSCode! This allows you to work on the cluster as if it were your local machine.

**Step 7:** **<span style="color: red;">IMPORTANT! Always close your deployment pod once you're done!</span>**
- Unlike jobs, deployments will keep running and don't shut down automatically.
- They keep blocking resources! (People will hate you for that, especially if you are using GPUs)

You can scale down your deployment to zero replicas to stop it:
```bash
kubectl scale deployment remote-deployment --replicas=0
```
or you can delete the deployment entirely:
```bash
kubectl delete deploy remote-deployment
```

- Next time you want to use it again, you can scale it back up:
```bash
kubectl scale deployment remote-deployment --replicas=1
```
or re-apply the deployment file:
```bash
kubectl apply -f remote-deployment.yml
```

### Fast command reference

```bash
kubectl get pods
```

```bash
kubectl get deploy
```

```bash
kubectl get pvc
```

## Workflows: prototyping vs training jobs

### Prototyping workflow

- Launch interactive pod
- Connect via port-forward + SSH/VS Code
- Explore, debug, and validate quickly

### Training/evaluation workflow

- Run scripted jobs with fixed configs
- Persist checkpoints and logs on PVC
- Track runs and metrics centrally

## Resource awareness

### GPU request example


In [ ]:
resources:
  requests:
    cpu: "1"
    nvidia.com/gpu: 1
  limits:
    nvidia.com/gpu: 1


### PVC example


In [ ]:
resources:
  requests:
    storage: 10Gi

## Experiment logging with WandB

### Why logging matters

- Compare experiments across parameter choices
- Track artifacts, metrics, and training curves
- Enable reproducibility and team collaboration

### Minimal setup


In [ ]:
pip install wandb
wandb login

In [ ]:
import wandb

wandb.init(project="dsw-2026", config={"lr": 1e-3, "batch_size": 32})
wandb.log({"train_loss": 0.42, "val_iou": 0.71})
wandb.finish()

## Exercise

- Run a script on the cluster
- Log metrics to WandB
- Share run link and short reflection on resource usage

## Legacy source mapping

- Existing cluster setup chapter: [mds_from_teo/lecture1_intro_k8s_docker_git.md](mds_from_teo/lecture1_intro_k8s_docker_git.md)
- Existing GPU workflow chapter: [mds_from_teo/lecture3_training_gpu.md](mds_from_teo/lecture3_training_gpu.md)
- Legacy cluster examples: [old-felix/cluster-examples/my-interactive-pod.yaml](old-felix/cluster-examples/my-interactive-pod.yaml), [old-felix/cluster-examples/my-data-pvc.yaml](old-felix/cluster-examples/my-data-pvc.yaml)